In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1. Cargar la matriz matemática
ruta_csv = '/content/drive/MyDrive/dataset_final_ml.csv'
df = pd.read_csv(ruta_csv)

# 2. Separar características (X) y el objetivo (y)
variable_objetivo = 'Limite_Elastico_YS_MPa'

X = df.drop(columns=[variable_objetivo])
y = df[variable_objetivo]

# 3. Partición: 80% entrenamiento, 20% examen (oculto)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 4. Instanciar los modelos (usamos n_jobs=-1 para usar todos los núcleos del CPU)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)

In [ ]:
# 5. Entrenamiento
print("Entrenando Random Forest...")
rf_model.fit(X_train, y_train)

print("Entrenando XGBoost...")
xgb_model.fit(X_train, y_train)

In [ ]:
# 6. Predicciones sobre el 20% de datos que nunca han visto
rf_pred = rf_model.predict(X_test)
xgb_pred = xgb_model.predict(X_test)

# 7. Evaluación de métricas
print("\n=== Resultados del Combate ===")
print(f"Random Forest -> R2: {r2_score(y_test, rf_pred):.4f} | RMSE: {np.sqrt(mean_squared_error(y_test, rf_pred)):.2f} MPa")
print(f"XGBoost       -> R2: {r2_score(y_test, xgb_pred):.4f} | RMSE: {np.sqrt(mean_squared_error(y_test, xgb_pred)):.2f} MPa")

# Explicabilidad

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Extraer importancias de ambos modelos
df_importancias = pd.DataFrame({
    'Variable': X.columns,
    'Importancia_RF': rf_model.feature_importances_,
    'Importancia_XGB': xgb_model.feature_importances_
})

# 2. Calcular el consenso (promedio) para definir el verdadero Top 15
df_importancias['Importancia_Promedio'] = df_importancias[['Importancia_RF', 'Importancia_XGB']].mean(axis=1)
top_15_consenso = df_importancias.sort_values(by='Importancia_Promedio', ascending=False).head(15)

# 3. Reestructurar la tabla para que Seaborn pueda poner barras comparativas
df_grafico = top_15_consenso.melt(
    id_vars='Variable',
    value_vars=['Importancia_RF', 'Importancia_XGB'],
    var_name='Modelo',
    value_name='Nivel de Importancia'
)

# 4. Generar el gráfico comparativo
plt.figure(figsize=(12, 8))
sns.barplot(x='Nivel de Importancia', y='Variable', hue='Modelo', data=df_grafico, palette='Set2')
plt.title('Intersección de Variables Físicas: Random Forest vs XGBoost')
plt.xlabel('Peso / Nivel de Importancia en el Modelo')
plt.ylabel('Descriptores')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

# Extraer importancias del modelo base
importancias_xgb = xgb_model.feature_importances_
indices_ordenados = np.argsort(importancias_xgb)[::-1]

# Barrido fino: de 10 a 100 variables, en pasos de 10
rango_variables = list(range(10, 110, 10))

errores_xgb = []
errores_rf = []

print("Calculando el valle de error. Esto tomará unos segundos...")

for n_vars in rango_variables:
    columnas_top = X.columns[indices_ordenados[:n_vars]]
    X_train_red = X_train[columnas_top]
    X_test_red = X_test[columnas_top]

    # Entrenar modelos rápidos
    xgb_tmp = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
    rf_tmp = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

    xgb_tmp.fit(X_train_red, y_train)
    rf_tmp.fit(X_train_red, y_train)

    # Registrar el RMSE
    errores_xgb.append(np.sqrt(mean_squared_error(y_test, xgb_tmp.predict(X_test_red))))
    errores_rf.append(np.sqrt(mean_squared_error(y_test, rf_tmp.predict(X_test_red))))

# Graficar el comportamiento
plt.figure(figsize=(10, 6))
plt.plot(rango_variables, errores_xgb, marker='o', linewidth=2, color='coral', label='XGBoost')
plt.plot(rango_variables, errores_rf, marker='s', linewidth=2, color='teal', label='Random Forest')

# Marcar nuestro punto dulce teórico
plt.axvline(x=50, color='gray', linestyle='--', alpha=0.7, label='Punto Dulce (50 Vars)')

plt.title('Curva de Rendimiento: Número de Variables vs RMSE')
plt.xlabel('Cantidad de Variables Retenidas (Top N)')
plt.ylabel('Error RMSE (MPa) - Más bajo es mejor')
plt.gca().invert_xaxis() # Invertimos el eje X para leerlo de izquierda (más) a derecha (menos)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

# Extraer importancias del modelo base
importancias_xgb = xgb_model.feature_importances_
indices_ordenados = np.argsort(importancias_xgb)[::-1]

# Micro-barrido: iterar de 50 a 71, de 1 en 1
rango_fino = list(range(50, 72, 1))
errores_micro = []

print("Iniciando escáner de alta precisión con lupa gráfica (50-70 variables)...")

for n_vars in rango_fino:
    # Seleccionar las N mejores columnas
    columnas_top = X.columns[indices_ordenados[:n_vars]]
    X_train_red = X_train[columnas_top]
    X_test_red = X_test[columnas_top]

    # Entrenar el motor
    xgb_tmp = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
    xgb_tmp.fit(X_train_red, y_train)

    # Calcular y guardar el RMSE
    rmse_actual = np.sqrt(mean_squared_error(y_test, xgb_tmp.predict(X_test_red)))
    errores_micro.append(rmse_actual)

# Encontrar al ganador absoluto
mejor_idx = np.argmin(errores_micro)
mejor_n = rango_fino[mejor_idx]
mejor_rmse = errores_micro[mejor_idx]

print(f"\n=== El Punto Dulce Absoluto ===")
print(f"Número óptimo de columnas a retener: {mejor_n}")
print(f"RMSE mínimo alcanzable en este valle: {mejor_rmse:.2f} MPa")

# ¡La grafiquita solicitada!
plt.figure(figsize=(10, 5))
plt.plot(rango_fino, errores_micro, marker='o', linestyle='-', color='coral', linewidth=2, markersize=6)

# Marcar el punto óptimo con una estrella gigante
plt.plot(mejor_n, mejor_rmse, markersize=18, color='red', label=f'Punto Dulce: {mejor_n} vars ({mejor_rmse:.1f} MPa)')
plt.axvline(x=mejor_n, color='crimson', linestyle='--', alpha=0.5)

plt.title('Micro-Barrido XGBoost: Zoom al Valle del Error (50-70 Variables)', fontsize=14)
plt.xlabel('Cantidad de Variables Retenidas', fontsize=12)
plt.ylabel('Error RMSE (MPa)', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
import os
import joblib
import json
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor

# Asegúrate de tener definida la variable objetivo de la celda 1
# variable_objetivo = 'Limite_Elastico_YS_MPa'

# 0. Crear la carpeta 'models' si no existe
os.makedirs('models', exist_ok=True)

# 1. Fijar el punto dulce descubierto
n_vars = 51
columnas_top_51 = X.columns[indices_ordenados[:n_vars]]

# 2. Construir el dataset de producción ultra-optimizado
X_train_final = X_train[columnas_top_51]
X_test_final = X_test[columnas_top_51]

# 3. Entrenar el modelo Campeón
print(f"Entrenando modelo XGBoost para '{variable_objetivo}' ({n_vars} variables)...")
xgb_final = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_final.fit(X_train_final, y_train)

# 4. Evaluar y calcular el rendimiento final
predicciones_finales = xgb_final.predict(X_test_final)
r2_final = float(r2_score(y_test, predicciones_finales))
rmse_final = float(np.sqrt(mean_squared_error(y_test, predicciones_finales)))

# Imprimir métricas en consola
print(f"\n=== Rendimiento del Modelo de Producción ===")
print(f"Variable Objetivo:       {variable_objetivo}")
print(f"R2 (Varianza Explicada): {r2_final:.4f}")
print(f"RMSE (Error Mínimo):     {rmse_final:.2f}\n")

# 5. Exportar el Modelo a la carpeta 'models'
nombre_modelo = f'models/xgboost_{variable_objetivo}.pkl'
joblib.dump(xgb_final, nombre_modelo)

# 6. Preparar el diccionario con la metadata completa (Métricas + Variables)
lista_variables = columnas_top_51.tolist()

metadata_exportacion = {
    "variable_objetivo": variable_objetivo,
    "rendimiento": {
        "R2": round(r2_final, 4),
        "RMSE": round(rmse_final, 2)
    },
    "cantidad_variables": n_vars,
    "variables_entrada": lista_variables
}

# 7. Exportar como JSON en la carpeta 'models'
nombre_json = f'models/variables_{variable_objetivo}.json'
with open(nombre_json, 'w') as f:
    json.dump(metadata_exportacion, f, indent=4)

# Confirmación de guardado e impresión de variables en consola
print(f"¡Éxito! Modelo exportado como: '{nombre_modelo}'")
print(f"¡Metadata exportada como: '{nombre_json}'\n")

print(f"=== Variables de Entrada Requeridas ({n_vars}) ===")
for i, var in enumerate(lista_variables, 1):
    print(f"{i}. {var}")